# Финальное решение

По суточной истории каждой куки строю признаки и усредняю оценки LightGBM и CatBoost
с весами 0.5 / 0.5. Обе модели обучаются на всём `train.csv`.
Параметры и набор признаков оставлены такими же, как в отправке с результатом **0.76000** на Stepik.

Ноутбук запускается отдельно от EDA и экспериментов. Результат в `outputs/submission.csv`.

In [1]:
from pathlib import Path
import sys
import platform

import numpy as np
import pandas as pd

ROOT = Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"
OUTPUTS = ROOT / "outputs"
SEED = 2026
THREADS = 4

from features import META, MISSING, prepare_events, get_event_names, make_features

print("Python", platform.python_version())

Python 3.13.5


## Данные

In [2]:
def read_meta(filename):
    frame = pd.read_csv(DATA / filename, dtype={"cookie_id": str})
    for column in META[1:]:
        frame[column] = pd.to_datetime(frame[column], utc=True)
    return frame

train = read_meta("train.csv")
train = train.sort_values(["window_start_ts", "cookie_id"], kind="stable").reset_index(drop=True)
test = read_meta("test.csv")
raw_events = pd.read_csv(DATA / "events.csv.gz", dtype=str)
meta = pd.concat([train[META], test[META]], ignore_index=True)

pd.Series({"train": len(train), "test": len(test), "events": len(raw_events)}, name="rows")

train      11091
test        4909
events    328905
Name: rows, dtype: int64

In [3]:
assert train.cookie_id.notna().all() and train.cookie_id.is_unique
assert test.cookie_id.notna().all() and test.cookie_id.is_unique
assert set(train.cookie_id).isdisjoint(test.cookie_id)
assert train.target.notna().all() and set(train.target) == {0, 1}
assert "target" not in test
assert meta[META[1:]].notna().all().all()
assert (meta.window_end_ts - meta.window_start_ts).dt.total_seconds().eq(86400).all()
assert (meta.cookie_created_at < meta.window_end_ts).all()
assert train.window_end_ts.max() <= test.window_start_ts.min()

print("Идентификаторы, разметка и окна наблюдения в порядке")

Идентификаторы, разметка и окна наблюдения в порядке


## Признаки

Оставляю события внутри `[window_start_ts, window_end_ts)` и удаляю точные дубликаты.
Словарь событий фиксирую по обучающей части первого временного разбиения  так же, как в экспериментах.
Сами агрегаты считаются отдельно для каждой куки.

В модели 597 признаков: действия, интервалы, сессии, разнообразие интересов,
поиск, нормализованный браузерный контекст и относительная геометрия курсора.
Неопределённые числовые статистики остаются `NaN`.

In [4]:
events = prepare_events(raw_events, meta)
assert events.event_ts.ge(events.window_start_ts).all()
assert events.event_ts.lt(events.window_end_ts).all()

vocabulary_train = train.loc[train.window_end_ts <= pd.Timestamp("2026-04-11", tz="UTC")]
event_names = get_event_names(vocabulary_train, events)
X = make_features(meta, events, event_names, n_jobs=THREADS)

X_train = X.loc[train.cookie_id].copy()
X_test = X.loc[test.cookie_id].copy()
y_train = train.target.to_numpy()

assert X_train.index.equals(pd.Index(train.cookie_id, name="cookie_id"))
assert X_test.index.equals(pd.Index(test.cookie_id, name="cookie_id"))
assert X_train.columns.equals(X_test.columns)
assert not {"cookie_id", "target", "window_start_ts", "window_end_ts"} & set(X.columns)
assert not np.isinf(X.select_dtypes("number").to_numpy()).any()

print("Обучение:", X_train.shape, "Тест:", X_test.shape)

Обучение: (11091, 597) Тест: (4909, 597)


## Обучение

LightGBM получает категориальные столбцы с категориями, определёнными на train.
Для CatBoost оставляю строки. Ни номера куки, ни сырые запросы и User-Agent в матрицу не входят.

In [5]:
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from threadpoolctl import threadpool_limits

categorical = X_train.select_dtypes("object").columns.tolist()
categories = {column: sorted(X_train[column].dropna().unique()) for column in categorical}

X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
for column, values in categories.items():
    X_train_lgb[column] = pd.Categorical(X_train_lgb[column], categories=values)
    X_test_lgb[column] = pd.Categorical(X_test_lgb[column], categories=values)

lgb = LGBMClassifier(
    n_estimators=500,
    num_leaves=15,
    min_child_samples=30,
    learning_rate=0.03,
    reg_lambda=10,
    colsample_bytree=0.85,
    random_state=SEED,
    n_jobs=THREADS,
    deterministic=True,
    force_col_wise=True,
    verbosity=-1,
)
cat = CatBoostClassifier(
    iterations=800,
    depth=5,
    learning_rate=0.035,
    l2_leaf_reg=20,
    loss_function="Logloss",
    cat_features=categorical,
    random_seed=SEED,
    thread_count=THREADS,
    allow_writing_files=False,
    verbose=False,
)

In [6]:
with threadpool_limits(limits=THREADS):
    lgb.fit(X_train_lgb, y_train)
    cat.fit(X_train, y_train)
    p_lgb = lgb.predict_proba(X_test_lgb)[:, 1]
    p_cat = cat.predict_proba(X_test)[:, 1]

score = 0.5 * p_lgb + 0.5 * p_cat

## Файл для отправки

Порог не выбираю: на платформу идут непрерывные оценки.
Проверки относятся только к формату и соответствию тестовым куки.

In [7]:
submission = pd.DataFrame({"cookie_id": test.cookie_id.to_numpy(), "score": score})
assert submission.columns.tolist() == ["cookie_id", "score"]
assert len(submission) == len(test)
assert submission.cookie_id.notna().all() and submission.cookie_id.is_unique
assert set(submission.cookie_id) == set(test.cookie_id)
assert np.isfinite(submission.score).all()
assert submission.score.between(0, 1).all()
submission.head()

,cookie_id,score
0,ck_315fb710a0e371e7,0.001898
1,ck_a76ee3b3e3e522fd,0.076842
2,ck_94c9a4d382689e82,0.038651
3,ck_8eaf9509ad9462a0,0.004291
4,ck_9a88a5a989cb5bc6,0.013016


In [9]:
OUTPUTS.mkdir(exist_ok=True)
submission.to_csv(submission_path, index=False, float_format="%.17g")

saved = pd.read_csv(submission_path, dtype={"cookie_id": str}, float_precision="round_trip")
assert np.array_equal(saved.score.to_numpy(), score)
assert saved.cookie_id.equals(submission.cookie_id)
print("Создан outputs/submission.csv:", len(saved), "строк")

Создан outputs/submission.csv: 4909 строк


**Результат отправки на Stepik: Precision@Recall≥0.70 = 0.76000.**
Это оценка платформы для сохранённой версии, а не метрика, вычисленная в этом ноутбуке:
тестовые метки недоступны. Константный ответ 0.09391.